# Marketing Attribution Model

**Digital Analytics | Marketing Analytics | Customer Journey Analytics**

This project compares first-touch, last-touch, linear, position-based, and time-decay attribution models on realistic ecommerce journey data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)


## Load Data


In [ ]:
touchpoints = pd.read_csv('../data/customer_journey_touchpoints.csv')
conversions = pd.read_csv('../data/conversions.csv')
print(touchpoints.shape)
print(conversions.shape)
touchpoints.head()


## Exploratory Analysis


In [ ]:
touchpoints['channel'].value_counts().plot(kind='bar', figsize=(10,5))
plt.title('Touchpoints by Marketing Channel')
plt.xlabel('Channel')
plt.ylabel('Touchpoints')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
conversions['touchpoints_before_conversion'].value_counts().sort_index().plot(kind='bar', figsize=(8,5))
plt.title('Conversion Path Length Distribution')
plt.xlabel('Touchpoints Before Conversion')
plt.ylabel('Conversions')
plt.tight_layout()
plt.show()


## Build Attribution Models


In [ ]:
df = touchpoints[touchpoints['converted'] == 1].copy()

df['first_touch_credit'] = (df['touchpoint_position'] == 1).astype(float)
df['last_touch_credit'] = (df['touchpoint_position'] == df['total_touchpoints']).astype(float)
df['linear_credit'] = 1 / df['total_touchpoints']

df['position_based_credit'] = 0.0
single = df['total_touchpoints'] == 1
multi = df['total_touchpoints'] > 1
df.loc[single, 'position_based_credit'] = 1
df.loc[multi & (df['touchpoint_position'] == 1), 'position_based_credit'] = .4
df.loc[multi & (df['touchpoint_position'] == df['total_touchpoints']), 'position_based_credit'] = .4
middle = multi & (df['touchpoint_position'] > 1) & (df['touchpoint_position'] < df['total_touchpoints'])
df.loc[middle, 'position_based_credit'] = .2 / (df.loc[middle, 'total_touchpoints'] - 2).replace(0, np.nan)
df['position_based_credit'] = df['position_based_credit'].fillna(0)

df['time_decay_raw'] = 0.5 ** (df['total_touchpoints'] - df['touchpoint_position'])
df['time_decay_credit'] = df['time_decay_raw'] / df.groupby('conversion_id')['time_decay_raw'].transform('sum')

for model in ['first_touch', 'last_touch', 'linear', 'position_based', 'time_decay']:
    df[f'{model}_revenue'] = df[f'{model}_credit'] * df['revenue_gbp']

df.head()


## Attribution Summary


In [ ]:
summary_frames = []
for model in ['first_touch', 'last_touch', 'linear', 'position_based', 'time_decay']:
    temp = df.groupby('channel').agg(
        attributed_revenue_gbp=(f'{model}_revenue', 'sum'),
        attributed_conversions=(f'{model}_credit', 'sum'),
        touches=('touchpoint_id', 'count'),
        media_cost_gbp=('media_cost_gbp', 'sum')
    ).reset_index()
    temp['model'] = model
    temp['roas'] = temp['attributed_revenue_gbp'] / temp['media_cost_gbp']
    summary_frames.append(temp)
model_summary = pd.concat(summary_frames, ignore_index=True).round(2)
model_summary.head(20)


## Compare Models


In [ ]:
pivot = model_summary.pivot(index='channel', columns='model', values='attributed_revenue_gbp')
pivot.plot(kind='bar', figsize=(12,6))
plt.title('Attributed Revenue by Channel and Model')
plt.xlabel('Channel')
plt.ylabel('Attributed Revenue GBP')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
time_decay_summary = model_summary[model_summary['model'] == 'time_decay'].sort_values('roas', ascending=False)
time_decay_summary.plot(x='channel', y='roas', kind='bar', figsize=(10,5), legend=False)
plt.title('Time-Decay Attribution ROAS by Channel')
plt.xlabel('Channel')
plt.ylabel('ROAS')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
time_decay_summary


## Business Recommendations

### Paid Search
Strong lower-funnel closer. Increase budget for high-intent campaigns while monitoring CPA and ROAS.

### Organic Search
Supports discovery and consideration. Invest in SEO landing pages and high-intent product content.

### Paid Social
Often assists early journey stages. Measure assisted conversions, not just last click.

### Email
Strong retention and conversion support. Expand basket recovery, loyalty, and lifecycle journeys.

### Display
Useful for awareness and retargeting but may be inefficient at broad prospecting. Tighten audience targeting.

### Affiliate
Review voucher and cashback partners for incrementality.

### Direct
Often captures demand created by previous touchpoints. Do not treat it as a pure acquisition channel.


In [ ]:
df.to_csv('../reports/touchpoint_attribution_output_from_notebook.csv', index=False)
model_summary.to_csv('../reports/attribution_model_summary_from_notebook.csv', index=False)
print('Outputs saved successfully.')
